In [1]:
# Load the clean dataset

import pandas as pd
df = pd.read_csv("../data/clean/imdb_clean.csv", dtype={"gross_revenue": "Int64"})
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   movie_id         1000 non-null   int64  
 1   movie_title      1000 non-null   str    
 2   released_year    1000 non-null   int64  
 3   certificate      1000 non-null   str    
 4   runtime          1000 non-null   int64  
 5   genre            1000 non-null   str    
 6   imdb_rating      1000 non-null   float64
 7   meta_score       843 non-null    float64
 8   director         1000 non-null   str    
 9   star1            1000 non-null   str    
 10  star2            1000 non-null   str    
 11  star3            1000 non-null   str    
 12  star4            1000 non-null   str    
 13  number_of_votes  1000 non-null   int64  
 14  gross_revenue    831 non-null    Int64  
 15  audience         1000 non-null   str    
dtypes: Int64(1), float64(2), int64(4), str(9)
memory usage: 126.1 KB


In [2]:
# directors table
directors = pd.DataFrame({"director_name": sorted(df["director"].unique())})
directors.insert(0, "director_id", range(1, len(directors) + 1))
directors.head()

,director_id,director_name
0,1,Aamir Khan
1,2,Aaron Sorkin
2,3,Abdellatif Kechiche
3,4,Abhishek Chaubey
4,5,Abhishek Kapoor


In [3]:
# certificates table
certificates = (df[["certificate", "audience"]]
                .drop_duplicates()
                .sort_values("certificate")
                .rename(columns={"certificate": "certificate_name"})
                .reset_index(drop=True))
certificates.insert(0, "certificate_id", range(1, len(certificates) + 1))
certificates

,certificate_id,certificate_name,audience
0,1,16,Adult
1,2,A,Adult
2,3,Approved,All ages
3,4,G,All ages
4,5,GP,Family/Teen
5,6,PG,Family/Teen
6,7,PG-13,Family/Teen
7,8,Passed,All ages
8,9,R,Adult
9,10,TV-14,Family/Teen


In [4]:
# Split the genre column into one row per movie and genre
movie_genre_names = df[["movie_id", "genre"]].copy()
movie_genre_names["genre_name"] = movie_genre_names["genre"].str.split(",")
movie_genre_names = movie_genre_names.explode("genre_name")
movie_genre_names["genre_name"] = movie_genre_names["genre_name"].str.strip()
movie_genre_names.head()

,movie_id,genre,genre_name
0,1,Drama,Drama
1,2,"Crime, Drama",Crime
1,2,"Crime, Drama",Drama
2,3,"Action, Crime, Drama",Action
2,3,"Action, Crime, Drama",Crime


In [5]:
# genres table
genres = pd.DataFrame({"genre_name": sorted(movie_genre_names["genre_name"].unique())})
genres.insert(0, "genre_id", range(1, len(genres) + 1))
genres

,genre_id,genre_name
0,1,Action
1,2,Adventure
2,3,Animation
3,4,Biography
4,5,Comedy
5,6,Crime
6,7,Drama
7,8,Family
8,9,Fantasy
9,10,Film-Noir


In [6]:
# Turn star1-star4 into one row per movie and actor; billing_order keeps the position (star1 -> 1)
cast_names = df.melt(id_vars="movie_id",
                     value_vars=["star1", "star2", "star3", "star4"],
                     var_name="billing_order",
                     value_name="actor_name")
cast_names["billing_order"] = cast_names["billing_order"].str.replace("star", "").astype(int)
cast_names.head()

,movie_id,billing_order,actor_name
0,1,1,Tim Robbins
1,2,1,Marlon Brando
2,3,1,Christian Bale
3,4,1,Al Pacino
4,5,1,Henry Fonda


In [7]:
# Movies where the same actor is listed twice
cast_names[cast_names.duplicated(["movie_id", "actor_name"], keep=False)].sort_values(["movie_id", "billing_order"])

,movie_id,billing_order,actor_name
120,121,1,Gene Kelly
1120,121,2,Gene Kelly
320,321,1,Buster Keaton
1320,321,2,Buster Keaton
727,728,1,Taika Waititi
2727,728,3,Taika Waititi
815,816,1,Massimo Troisi
1815,816,2,Massimo Troisi


In [8]:
# Keep each actor once per movie, with their highest billing
cast_names = (cast_names.sort_values(["movie_id", "billing_order"])
                        .drop_duplicates(["movie_id", "actor_name"], keep="first"))
len(cast_names)

3996

In [9]:
# actors table
actors = pd.DataFrame({"actor_name": sorted(cast_names["actor_name"].unique())})
actors.insert(0, "actor_id", range(1, len(actors) + 1))
actors.head()

,actor_id,actor_name
0,1,Aamir Bashir
1,2,Aamir Khan
2,3,Aaron Eckhart
3,4,Aaron Taylor-Johnson
4,5,Abdel Ahmed Ghili


In [10]:
# movies table: replace director and certificate names with director_id and certificate_id (FKs)
movies = (df.merge(directors, left_on="director", right_on="director_name")
            .merge(certificates, left_on=["certificate", "audience"],
                   right_on=["certificate_name", "audience"])
            .sort_values("movie_id"))

movies = movies[["movie_id", "movie_title", "released_year", "runtime",
                 "gross_revenue", "director_id", "certificate_id"]].reset_index(drop=True)
movies.head()

,movie_id,movie_title,released_year,runtime,gross_revenue,director_id,certificate_id
0,1,The Shawshank Redemption,1994,142,28341469,142,2
1,2,The Godfather,1972,175,134966411,138,2
2,3,The Dark Knight,2008,152,534858444,84,15
3,4,The Godfather: Part II,1974,202,57300000,138,2
4,5,12 Angry Men,1957,96,4360000,457,13


In [11]:
# ratings table
ratings = df[["movie_id", "imdb_rating", "meta_score", "number_of_votes"]].copy()
ratings.insert(0, "rating_id", range(1, len(ratings) + 1))
ratings = ratings[["rating_id", "imdb_rating", "meta_score", "number_of_votes", "movie_id"]]
ratings.head()

,rating_id,imdb_rating,meta_score,number_of_votes,movie_id
0,1,9.3,80.0,2343110,1
1,2,9.2,100.0,1620367,2
2,3,9.0,84.0,2303232,3
3,4,9.0,90.0,1129952,4
4,5,9.0,96.0,689845,5


In [12]:
# movie_genres table
movie_genres = (movie_genre_names.merge(genres, on="genre_name")
                .sort_values(["movie_id", "genre_id"])
                [["movie_id", "genre_id"]]
                .reset_index(drop=True))
movie_genres.insert(0, "movie_genre_id", range(1, len(movie_genres) + 1))
movie_genres.head()

,movie_genre_id,movie_id,genre_id
0,1,1,7
1,2,2,6
2,3,2,7
3,4,3,1
4,5,3,6


In [13]:
# movie_cast table
movie_cast = (cast_names.merge(actors, on="actor_name")
              .sort_values(["movie_id", "billing_order"])
              [["movie_id", "actor_id", "billing_order"]]
              .reset_index(drop=True))
movie_cast.insert(0, "movie_cast_id", range(1, len(movie_cast) + 1))
movie_cast.head()

,movie_cast_id,movie_id,actor_id,billing_order
0,1,1,2483,1
1,2,1,1832,2
2,3,1,279,3
3,4,1,2641,4
4,5,2,1664,1


In [14]:
# Check primary keys are unique
for name, table, pk in [("directors", directors, "director_id"),
                        ("certificates", certificates, "certificate_id"),
                        ("genres", genres, "genre_id"),
                        ("actors", actors, "actor_id"),
                        ("movies", movies, "movie_id"),
                        ("ratings", ratings, "rating_id"),
                        ("movie_genres", movie_genres, "movie_genre_id"),
                        ("movie_cast", movie_cast, "movie_cast_id")]:
    print(name, "- duplicate PKs:", table[pk].duplicated().sum())

directors - duplicate PKs: 0
certificates - duplicate PKs: 0
genres - duplicate PKs: 0
actors - duplicate PKs: 0
movies - duplicate PKs: 0
ratings - duplicate PKs: 0
movie_genres - duplicate PKs: 0
movie_cast - duplicate PKs: 0


In [15]:
# Check every foreign key exists in its parent table
print("movies.director_id:", (~movies["director_id"].isin(directors["director_id"])).sum())
print("movies.certificate_id:", (~movies["certificate_id"].isin(certificates["certificate_id"])).sum())
print("ratings.movie_id:", (~ratings["movie_id"].isin(movies["movie_id"])).sum())
print("movie_genres.movie_id:", (~movie_genres["movie_id"].isin(movies["movie_id"])).sum())
print("movie_genres.genre_id:", (~movie_genres["genre_id"].isin(genres["genre_id"])).sum())
print("movie_cast.movie_id:", (~movie_cast["movie_id"].isin(movies["movie_id"])).sum())
print("movie_cast.actor_id:", (~movie_cast["actor_id"].isin(actors["actor_id"])).sum())

movies.director_id: 0
movies.certificate_id: 0
ratings.movie_id: 0
movie_genres.movie_id: 0
movie_genres.genre_id: 0
movie_cast.movie_id: 0
movie_cast.actor_id: 0


In [16]:
# Check no movie has the same genre or actor twice, and no movie was lost
print(movie_genres.duplicated(["movie_id", "genre_id"]).sum())
print(movie_cast.duplicated(["movie_id", "actor_id"]).sum())
print(len(movies), len(ratings))

0
0
1000 1000


In [17]:
# Save each table as a CSV in data/clean (tables without FKs first, the order to load them in SQL)
tables = {"directors": directors,
          "certificates": certificates,
          "genres": genres,
          "actors": actors,
          "movies": movies,
          "ratings": ratings,
          "movie_genres": movie_genres,
          "movie_cast": movie_cast}

for name, table in tables.items():
    table.to_csv(f"../data/clean/{name}.csv", index=False)
    print(f"{name}.csv saved - {len(table)} rows")

directors.csv saved - 548 rows
certificates.csv saved - 16 rows
genres.csv saved - 21 rows
actors.csv saved - 2709 rows
movies.csv saved - 1000 rows
ratings.csv saved - 1000 rows
movie_genres.csv saved - 2541 rows
movie_cast.csv saved - 3996 rows
